In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

In [2]:
# Authoritative paths from project context
DATA_ROOT = Path('/resnick/groups/CS156b/from_central/data')
TRAIN_IMG_ROOT = DATA_ROOT / 'train'
TRAIN_CSV = DATA_ROOT / 'student_labels' / 'train2023.csv'
TEST_IMG_ROOT = DATA_ROOT / 'test'
TEST_IDS_CSV = DATA_ROOT / 'student_labels' / 'test_ids.csv'

TEAM_ROOT = Path('/resnick/groups/CS156b/from_central/2026/haa')
EFFICIENT_OUT = TEAM_ROOT / 'efficient_net_data'
MANIFEST_DIR = EFFICIENT_OUT / 'manifests'

for p in [TRAIN_IMG_ROOT, TRAIN_CSV, TEST_IMG_ROOT, TEST_IDS_CSV]:
    print(f'{p}:', 'OK' if p.exists() else 'MISSING')

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
print('Manifest output dir:', MANIFEST_DIR)

/resnick/groups/CS156b/from_central/data/train: OK
/resnick/groups/CS156b/from_central/data/student_labels/train2023.csv: OK
/resnick/groups/CS156b/from_central/data/test: OK
/resnick/groups/CS156b/from_central/data/student_labels/test_ids.csv: OK
Manifest output dir: /resnick/groups/CS156b/from_central/2026/haa/efficient_net_data/manifests


In [ ]:
LABEL_COLS = [
    'No Finding',
    'Enlarged Cardiomediastinum',
    'Cardiomegaly',
    'Lung Opacity',
    'Pneumonia',
    'Pleural Effusion',
    'Pleural Other',
    'Fracture',
    'Support Devices',
]

df = pd.read_csv(TRAIN_CSV)
df['abs_path'] = df['Path'].astype(str).str.replace('^train/', str(TRAIN_IMG_ROOT) + '/', regex=True)
df['is_frontal'] = df['Path'].astype(str).str.contains('frontal', case=False, na=False)

display(df.head(3))
print('Rows:', len(df))
print('Frontal rows:', int(df['is_frontal'].sum()))

In [ ]:
# Quick label value audit: expected values commonly include {1, 0, -1, NaN}
for c in LABEL_COLS:
    vc = df[c].value_counts(dropna=False).sort_index()
    print('\n' + c)
    print(vc)

In [ ]:
# Build a frontal-only training manifest (paths + labels), no image copy
frontal = df[df['is_frontal']].copy()

# Keep labels numeric; preserve NaN for masking in training code if desired
manifest_cols = ['Path', 'abs_path'] + LABEL_COLS
train_manifest = frontal[manifest_cols].reset_index(drop=True)

manifest_path = MANIFEST_DIR / 'train_frontal_manifest.csv'
train_manifest.to_csv(manifest_path, index=False)
print('Wrote:', manifest_path, 'rows=', len(train_manifest))

In [ ]:
# Build test manifest from test IDs CSV, no image copy
test_ids = pd.read_csv(TEST_IDS_CSV)
test_ids['abs_path'] = test_ids['Path'].astype(str).str.replace('^test/', str(TEST_IMG_ROOT) + '/', regex=True)

test_manifest_path = MANIFEST_DIR / 'test_manifest.csv'
test_ids[['Path', 'abs_path']].to_csv(test_manifest_path, index=False)
print('Wrote:', test_manifest_path, 'rows=', len(test_ids))

## Notes
- Raw data remains in read-only course storage; we do not copy raw data into the repo.
- Manifests are written outside the repo to `/resnick/groups/CS156b/from_central/2026/haa/efficient_net_data/manifests`.
- During training, load each image from `abs_path` on demand.
- If the cluster path for `test_ids.csv` differs, update `TEST_IDS_CSV` in one place above.